# MNIST Digits Recognition - MLOps

## Kubeflow pipeline

In [1]:
#needs to be run if notebook has been restarted
!pip install tensorflow

  Using cached tensorflow-2.18.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-24.3.25-py2.py3-none-any.whl.metadata (850 bytes)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-manylinux2010_x86_64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached tensorboard-2.18.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached keras-3.6.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached numpy-2.0.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
  Using cached ml_dtypes-0.4.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
  Using cached tensorflow_io_gcs_filesystem-0.37.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.meta

In [2]:
#imports
import os
import kfp
from pathlib import Path
from kfp import dsl
from kfp.dsl import Input, Output, Dataset, Model, Metrics, ClassificationMetrics
import numpy as np
from tensorflow import keras
import tensorflow as tf
import glob
from sklearn.metrics import confusion_matrix

from kubernetes import client 
from kserve import KServeClient
from kserve import constants
from kserve import utils
from kserve import V1beta1InferenceService
from kserve import V1beta1InferenceServiceSpec
from kserve import V1beta1PredictorSpec
from kserve import V1beta1TFServingSpec

from datetime import datetime
import time
import subprocess
import sys
from kale.common.serveutils import serve
import mlflow
import mlflow.keras
from mlflow.models import ModelSignature
from mlflow.types import Schema, ColSpec, TensorSpec
import numpy as np
import itertools


2024-10-24 15:20:29.420350: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-10-24 15:20:29.421943: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-10-24 15:20:29.425720: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-10-24 15:20:29.434515: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-24 15:20:29.448128: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been 

In [3]:
#set MLFlow Experiment
experiment_name= 'mlops'
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='s3://mlflow.rack2-mil/3', creation_time=1729271579728, experiment_id='3', last_update_time=1729271579728, lifecycle_stage='active', name='mlops', tags={}>

In [1]:
# get the directory
#kfp_client = kfp.Client()
#user_mounted_dir_name = kfp_client.get_user_namespace()
#user_shared_dir = f"{str(Path.home())}/{user_mounted_dir_name}/"
#user_shared_dir

In [6]:
def pip_install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])
    
#create function that can update the jwt token to run a long mlflow experiment
def create_token():
    # Directly call the magic function to update the token
    get_ipython().run_line_magic("update_token", "")
    return os.environ.get("MLFLOW_TRACKING_TOKEN")

#wrapper function for mlflow logging that checks for valid token
def __call_and_check_token(fun, *args):
    try:
        return fun(*args)
    except Exception as e:
        print(f"Exception: {e}. Token probably expired. Creating a new token...")
        os.environ["MLFLOW_TRACKING_TOKEN"] = create_token()
        return fun(*args)

#function for checking for jwt token and updating it before running the next mlflow run
def start_mlflow_run():
    try:
        return mlflow.start_run()
    except Exception as e:
        print(f"Exception: {e}. Token probably expired. Creating a new token...")
        os.environ["MLFLOW_TRACKING_TOKEN"] = create_token()
        return mlflow.start_run()


In [5]:
# Specify proxy and no_proxy value if applicable
http_proxy="" ## "http://<proxy_host>:<port>"
# After default.svc append comma separated list of IP address, hostname, CIDR block, service name, etc.
no_proxy="" ## "localhost,.cluster.local,.svc,.default.svc, <host>:<port>, x.x.x.x/y,..."

#set directories for data
user_shared_dir = "/mnt/data/"
training_data_subdir = "training/"
model_subdir = "model"
model_trained_subdir = "model_trained"
export_model_dir = user_shared_dir +"export/"


In [7]:
#import and format data
final_training_data_dir = user_shared_dir + training_data_subdir

x_train_artifact = final_training_data_dir + "train-images-pickle.npy"
x_test_artifact = final_training_data_dir + "test-images-pickle.npy"
y_train_artifact = final_training_data_dir + "train-labels-pickle.npy"
y_test_artifact = final_training_data_dir + "test-labels-pickle.npy"


x_train_preproc = final_training_data_dir + "xtrain-preproc.npy"
x_test_preproc = final_training_data_dir + "xtest-preproc.npy"

# load data artifact store
x_train = np.load(x_train_artifact) 
x_test = np.load(x_test_artifact)
    
# reshaping the data
# reshaping pixels in a 28x28px image with greyscale, canal = 1. This is needed for the Keras API
x_train = x_train.reshape(-1,28,28,1)
x_test = x_test.reshape(-1,28,28,1)
# normalizing the data
# each pixel has a value between 0-255. Here we divide by 255, to get values from 0-1
x_train = x_train / 255
x_test = x_test / 255
    
#logging metrics using Kubeflow Artifacts
print("Len x_train", x_train.shape[0])
print("Len y_train", x_test.shape[0])
      
# save feature in artifact store
np.save(x_train_preproc, x_train)
#os.rename("tmp/x_train.npy", x_train_processed.path)
    
np.save(x_test_preproc, x_test)
#os.rename("tmp/x_test.npy", x_test_processed.path)

Len x_train 60000
Len y_train 10000


In [8]:
# Define the directory and filename for saving the model
final_model_data_dir = os.path.join(user_shared_dir, model_subdir)
model_file_path = os.path.join(final_model_data_dir, "model.keras")  # Use .keras or .h5 based on error

# Create the directory if it doesn't exist
if not os.path.exists(final_model_data_dir):
    os.makedirs(final_model_data_dir, 0o777)

# Build your model 
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=(28, 28, 1)))  # Define the input shape 
model.add(keras.layers.Conv2D(64, (3, 3), activation='relu'))
model.add(keras.layers.MaxPool2D(2, 2))
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(64, activation='relu'))
model.add(keras.layers.Dense(32, activation='relu'))
model.add(keras.layers.Dense(10, activation='softmax'))

# Save the model
keras.models.save_model(model, model_file_path)


In [9]:
print("Model file path:", model_file_path)

Model file path: /home/eva.murdock-hpe.com/eva-murdock-hpe-com-534e267b/model/model.keras


In [10]:
# Define directories and model file path
model_trained_dir = user_shared_dir + model_trained_subdir
#model_file_path = 'eva-murdock-hpe-com-534e267b/model/model.keras' #os.path.join(model_trained_dir, "model.keras")

if not os.path.exists(model_trained_dir):
    os.makedirs(model_trained_dir, 0o777)

In [11]:
# Load dataset
x_train = np.load(x_train_preproc)
x_test = np.load(x_test_preproc)
y_train = np.load(y_train_artifact)
y_test = np.load(y_test_artifact)

In [12]:
# Load model structure
model = keras.models.load_model(model_file_path)  # Load the model with the .keras extension


In [15]:
# Define input and output schemas
input_schema = Schema([
    TensorSpec(shape=(-1, 28, 28, 1), type=np.dtype('float32'), name="input_1") 
])
output_schema = Schema([
    TensorSpec(shape=(None, 10), type=np.dtype('float32'), name="output_1")  
])

# Create model signature
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

In [16]:
#specify parameters to be tuned on
parameters = {
    "num_epochs": [1, 2],
    "learning_rate": [0.1, 0.2],
    # "verbose": True,
}

# generate parameters combinations
params_keys = parameters.keys()
params_values = [
    parameters[key] if isinstance(parameters[key], list) else [parameters[key]]
    for key in params_keys
]
runs_parameters = [
    dict(zip(params_keys, combination)) for combination in itertools.product(*params_values)
]

In [17]:
# Iterate over parameter combinations
for i, run_parameters in enumerate(runs_parameters):
    print(f"Run {i}: {run_parameters}")

    # End any active MLflow run
    if mlflow.active_run():
        mlflow.end_run()

    # Start an MLflow run
    mlflow.start_run(run_name=f"Run {i}")

     # Compile the model using parameters from the run
    model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=run_parameters["learning_rate"]),
                  loss="sparse_categorical_crossentropy",
                  metrics=['accuracy'])

    # Log hyperparameters
    for key, value in run_parameters.items():
        __call_and_check_token(mlflow.log_param, key, value)

    # Fit the model
    history = model.fit(x=x_train, y=y_train, epochs=run_parameters["num_epochs"], batch_size=20)

    # Evaluate the model
    model_loss, model_accuracy = model.evaluate(x_test, y_test)

    # Log metrics
    __call_and_check_token(mlflow.log_metric, "model_loss", model_loss)
    __call_and_check_token(mlflow.log_metric, "model_accuracy", model_accuracy)

    # Predictions
    y_predict = model.predict(x_test)
    y_predict = np.argmax(y_predict, axis=1)

    # Save model in local storage
    keras.models.save_model(model, model_file_path)

    # Log the trained model in MLflow
    mlflow.keras.log_model(model, "model", signature=signature)

    # Adding /1/ subfolder for TFServing
    model_trained_dir = model_trained_dir + '/1/'
    print("model_trained_uri: " + model_trained_dir)

    export_model_dir = export_model_dir + '/1/'
    print("model_export_uri: " + model_trained_dir)

    # Create the directory if it doesn't exist for model export
    if not os.path.exists(export_model_dir):
        os.makedirs(export_model_dir, 0o777)

    # Specify the filename with the correct extension
    model_export_file_path = os.path.join(export_model_dir, "model.keras")

    # Save model in model export volume
    keras.models.save_model(model, model_export_file_path)

    # Evaluate the model on the test dataset
    test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=1)
    print('The testing accuracy is :', test_accuracy * 100, '%')

    # Log the test accuracy
    __call_and_check_token(mlflow.log_metric, "test_accuracy", test_accuracy)

    # Save predictions to a .npy file
    predictions_file = "predictions.npy"
    np.save(predictions_file, y_predict)  # Save predictions locally

    # Log the predictions file as an artifact
    __call_and_check_token(mlflow.log_artifact, predictions_file)

    # End the MLflow run
    mlflow.end_run()
    print("")


Run 0: {'num_epochs': 1, 'learning_rate': 0.1}
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 149s 50ms/step - accuracy: 0.8632 - loss: 0.4236
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.9700 - loss: 0.0923
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step


/opt/conda/lib/python3.11/site-packages/_distutils_hack/__init__.py:32: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


model_trained_uri: /home/eva.murdock-hpe.com/eva-murdock-hpe-com-534e267b/model_trained/1/
model_export_uri: /home/eva.murdock-hpe.com/eva-murdock-hpe-com-534e267b/model_trained/1/
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - accuracy: 0.9700 - loss: 0.0923
The testing accuracy is : 97.54999876022339 %

Run 1: {'num_epochs': 1, 'learning_rate': 0.2}
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 149s 49ms/step - accuracy: 0.9699 - loss: 0.0948
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 22ms/step - accuracy: 0.9782 - loss: 0.0683
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 21ms/step
model_trained_uri: /home/eva.murdock-hpe.com/eva-murdock-hpe-com-534e267b/model_trained/1//1/
model_export_uri: /home/eva.murdock-hpe.com/eva-murdock-hpe-com-534e267b/model_trained/1//1/
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 21ms/step - accuracy: 0.9782 - loss: 0.0683
The testing accuracy is : 98.17000031471252 %

Run 2: {'num_epochs': 2, 'learning_rate': 0.1}
Epoch 1/2
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 141s 47ms/step - accuracy: 0.9907 - loss: 0.0304
Epoc

In [19]:
# Load model from the volume
model = keras.models.load_model(model_export_file_path)

# Start MLflow to check test accuracy and make predictions run using the wrapper
with start_mlflow_run() as run:
    # Evaluate the model on the test dataset
    test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=1)
    print('The testing accuracy is :', test_accuracy * 100, '%')
    
    # Log the test accuracy using the wrapper
    __call_and_check_token(mlflow.log_metric, "test_accuracy", test_accuracy)
    
    # Make predictions
    preds = model.predict(x_test, verbose=1)
    print(preds)

    # Save predictions to a .npy file
    predictions_file = "predictions.npy"
    np.save(predictions_file, preds)  # Save locally

    # Log the predictions file as an artifact using the wrapper
    __call_and_check_token(mlflow.log_artifact, predictions_file)


313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 20ms/step - accuracy: 0.9825 - loss: 0.0624
The testing accuracy is : 98.61000180244446 %
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step
[[7.1454412e-11 1.2528574e-07 3.7935028e-08 ... 9.9999988e-01
  3.8671479e-09 4.0413735e-09]
 [1.6671813e-09 9.7692991e-09 1.0000000e+00 ... 3.5466458e-12
  5.5269542e-11 5.1936355e-12]
 [8.5546020e-08 9.9997377e-01 1.3759241e-06 ... 1.2538406e-05
  1.0447631e-06 5.3046892e-07]
 ...
 [4.9310890e-12 2.6723965e-10 1.5570791e-08 ... 1.1405156e-09
  9.5418100e-09 8.2521581e-09]
 [9.8090468e-12 5.3916174e-09 6.2767445e-13 ... 5.6335864e-10
  5.7552967e-05 2.5540103e-08]
 [4.0908513e-10 5.1270412e-12 5.8960836e-11 ... 3.6796949e-15
  2.0258668e-11 2.3992668e-13]]


In [ ]:
from kale.sdk import step
import subprocess

model_dir = "/mnt/export/"
model_volume = "model-volume-kale"  
model_name = "digitrec-serve"

api_version = 'serving.kserve.io/v1beta1'

serving_command = f"""
     echo '
      apiVersion: "{api_version}"
      kind: "InferenceService"
      metadata:
        name: {model_name}
        annotations:
          "sidecar.istio.io/inject": "false"
      spec:
        predictor:
          tensorflow:
            storageUri: "pvc://{model_volume}/"
            resources:
              limits:
                cpu: '2'
                memory: 1Gi
              requests:
                cpu: 100m
                memory: 50Mi'  | kubectl create -f -
      """

# Run the serving command
result = subprocess.run(serving_command, shell=True, check=True)
    
# Check if the serving container is running successfully
if result.returncode == 0:
    print("TensorFlow Serving started successfully.")
else:
    raise Exception("Failed to start TensorFlow Serving.")

### Next step: Inference test

Go the the following step: [Inference notebook](03-Inference.ipynb)